# Synthetic Control: Convex Counterfactuals and Placebo Inference

**Econometrics Notebook Library · v0.1.0**

## Intuition

When one aggregate unit is treated, a weighted combination of untreated units can approximate its pre-treatment path. Classical synthetic control chooses nonnegative donor weights that sum to one, making the counterfactual an interpolation inside the donor convex hull.

Reference: [Abadie, Diamond & Hainmueller, JASA (2010)](https://doi.org/10.1198/jasa.2009.ap08746).

## Optimization problem

Let $Y_{1,pre}$ be the treated unit's pre-period vector and $Y_{0,pre}$ the matrix of donor outcomes. A simple outcome-matching SCM solves

$$
\min_w \|Y_{1,pre}-Y_{0,pre}w\|_2^2
\quad\text{s.t.}\quad
w_j\ge0,\;\sum_j w_j=1.
$$

Post-treatment gaps are

$$\hat\tau_t=Y_{1t}-Y_{0t}'\hat w.$$

Inference is often design/placebo based rather than justified by a large-$N$ regression asymptotic.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.5)
pd.set_option("display.max_columns", 30)

In [ ]:
from econnotes.core import simulate_synthetic_control, synthetic_control_weights, scm_placebo_rmspe_ratios

sim = simulate_synthetic_control(n_donors=18, t_pre=20, t_post=10, tau=-2.0, seed=303)
treated, donors, t_pre = sim["treated"], sim["donors"], sim["t_pre"]
w = synthetic_control_weights(treated[:t_pre], donors[:, :t_pre].T)
synth = w @ donors
gap = treated - synth
{"sum_weights":w.sum(), "largest_weight":w.max(), "pre_RMSPE":np.sqrt(np.mean(gap[:t_pre]**2)),
 "mean_post_gap":gap[t_pre:].mean()}

In [ ]:
fig, ax = plt.subplots()
ax.plot(treated, label="Treated")
ax.plot(synth, label="Synthetic control")
ax.axvline(t_pre-0.5, linestyle="--", label="Treatment")
ax.set(xlabel="Time", ylabel="Outcome", title="Synthetic control counterfactual")
ax.legend();

In [ ]:
treated_ratio, placebo_ratios = scm_placebo_rmspe_ratios(treated, donors, t_pre)
p_placebo = (1 + np.sum(placebo_ratios >= treated_ratio))/(1 + len(placebo_ratios))
{"treated_post_pre_RMSPE_ratio":treated_ratio, "placebo_rank_p":p_placebo}

## Common failure

A dramatic post-treatment gap is not persuasive if pre-treatment fit is poor. A treated unit outside the donor convex hull forces SCM to approximate an extrapolation with interpolation weights; pre-fit diagnostics will often reveal the problem.

## Researcher failure checklist

- Report pre-treatment RMSPE and the donor weights.
- Exclude donors affected by the intervention or spillovers.
- Show in-space placebo gaps or RMSPE ratios, not just one treated-unit chart.
- Do not cherry-pick predictor windows after seeing the post-treatment fit.
- Discuss whether the treated unit lies plausibly inside the donor support/convex hull.